In [ ]:
import os
import numpy as np
import jpegio as jio

root = "jpeg_dataset"


def get_dequantized_dct_coefficients(jpeg, channel_index):
    coef_q = jpeg.coef_arrays[channel_index]

    # Y uses table 0, Cb/Cr use table 1
    qtbl_index = channel_index if channel_index == 0 else 1
    qtbl = jpeg.quant_tables[qtbl_index]

    H, W = coef_q.shape
    qtbl_full = np.tile(qtbl, (H // 8, W // 8))

    return coef_q * qtbl_full


def get_quantized_dct_coefficients(jpeg, channel_index):
    return jpeg.coef_arrays[channel_index]


for split in ["train", "test"]:

    split_dir = os.path.join(root, split)

    if not os.path.isdir(split_dir):
        continue

    for label in os.listdir(split_dir):

        label_dir = os.path.join(split_dir, label)

        if not os.path.isdir(label_dir):
            continue

        save_dir = os.path.join(
            "dct_dataset",
            split,
            label
        )

        os.makedirs(save_dir, exist_ok=True)

        for file in os.listdir(label_dir):

            if not file.endswith(".jpeg"):
                continue

            path = os.path.join(label_dir, file)

            jpeg = jio.read(path)

            dct = {
                "Y": get_dequantized_dct_coefficients(jpeg, 0),
                "Cb": get_dequantized_dct_coefficients(jpeg, 1),
                "Cr": get_dequantized_dct_coefficients(jpeg, 2)
            }

            np.save(
                os.path.join(
                    save_dir,
                    file.replace(".jpeg", ".npy")
                ),
                dct
            )